# A2：並行 Tool-Calling 與 DAG 依賴解析引擎

> **對應 Blog：** FDE 面試準備指南（二十二）動態並行 Tool-Calling 與依賴解析引擎  
> **核心目標：** 用 asyncio 實作真正的 DAG 依賴解析，親眼看到延遲從 T₁+T₂+T₃ 降到 max(T₁,T₂,T₃)

## 面試情境

> 「Gemini 判定需要同時呼叫三個工具：get_user_profile、get_order_history、get_risk_score。這三個工具執行時間不同。有時候工具 B 的輸入必須依賴工具 A 的輸出。你如何設計動態工具執行引擎最大化並行，並處理這種動態依賴關係？」

## 學習路徑

| Part | 內容 | 關鍵觀察 |
|------|------|----------|
| 1 | 順序執行（展示延遲代價） | 850ms 延遲 |
| 2 | 完全並行（無依賴的情況） | 降到 400ms |
| 3 | DAG 依賴解析引擎 | 處理有依賴的混合情況 |
| 4 | 錯誤處理策略 | Fail-fast vs Partial Success |
| 5 | 面試白板語言 | 量化結果 + trade-off |

In [ ]:
import asyncio
import time
import json
from typing import Any, Callable, Coroutine
from dataclasses import dataclass, field

print("asyncio 版本：", asyncio.__version__ if hasattr(asyncio, '__version__') else "內建")
print("環境就緒")

## Part 1：順序執行（展示延遲代價）

In [ ]:
# 模擬工具（帶真實延遲）

async def get_user_profile(user_id: str) -> dict:
    """模擬 API 呼叫：150ms"""
    await asyncio.sleep(0.15)
    return {
        "user_id": user_id,
        "name": "王大明",
        "tier": "GOLD",
        "credit_score": 750
    }

async def get_order_history(user_id: str) -> dict:
    """模擬資料庫查詢：400ms"""
    await asyncio.sleep(0.40)
    return {
        "user_id": user_id,
        "total_orders": 47,
        "total_value": 125_000,
        "last_30_days": 3
    }

async def get_risk_score(user_id: str, profile: dict = None) -> dict:
    """模擬風險計算：300ms（可選讀取 profile）"""
    await asyncio.sleep(0.30)
    # 如果有 profile，可以做更精準的風險評估
    credit_bonus = 0
    if profile and profile.get("credit_score", 0) > 700:
        credit_bonus = 10
    return {
        "user_id": user_id,
        "risk_score": 35 - credit_bonus,  # 越低越好
        "risk_level": "LOW"
    }

async def generate_offer(order_history: dict, risk: dict) -> dict:
    """生成報價：200ms（依賴 order_history 和 risk）"""
    await asyncio.sleep(0.20)
    discount = 5 if risk["risk_level"] == "LOW" else 0
    return {
        "offer_type": "PREMIUM" if order_history["total_orders"] > 20 else "STANDARD",
        "discount_pct": discount,
        "credit_limit": 50_000 if risk["risk_level"] == "LOW" else 10_000
    }

print("工具定義完成：")
print("  get_user_profile: 150ms")
print("  get_order_history: 400ms")
print("  get_risk_score: 300ms（可讀取 profile）")
print("  generate_offer: 200ms（依賴 order_history + risk）")

In [ ]:
# ❌ 順序執行（最差的方案）

async def sequential_execution(user_id: str) -> dict:
    """一個一個呼叫工具"""
    start = time.time()
    
    profile = await get_user_profile(user_id)
    print(f"  [{time.time()-start:.2f}s] get_user_profile 完成")
    
    orders = await get_order_history(user_id)
    print(f"  [{time.time()-start:.2f}s] get_order_history 完成")
    
    risk = await get_risk_score(user_id, profile)
    print(f"  [{time.time()-start:.2f}s] get_risk_score 完成")
    
    offer = await generate_offer(orders, risk)
    print(f"  [{time.time()-start:.2f}s] generate_offer 完成")
    
    total = time.time() - start
    print(f"\n⏱️  順序執行總時間: {total:.2f}s")
    print(f"   理論最小值: 0.15 + 0.40 + 0.30 + 0.20 = 1.05s")
    return {"profile": profile, "orders": orders, "risk": risk, "offer": offer}

print("=" * 50)
print("❌ 順序執行")
print("=" * 50)
result = await sequential_execution("user_001")

## Part 2：完全並行（無依賴的情況）

In [ ]:
# ✅ 簡單並行（三個獨立工具同時執行）

async def simple_parallel(user_id: str) -> dict:
    """asyncio.gather 並行執行獨立工具"""
    start = time.time()
    
    # 同時啟動三個獨立工具
    profile, orders, risk = await asyncio.gather(
        get_user_profile(user_id),
        get_order_history(user_id),
        get_risk_score(user_id)  # 這裡不傳 profile（假設獨立）
    )
    print(f"  [{time.time()-start:.2f}s] 三個工具全部完成（並行）")
    
    # generate_offer 依賴 orders 和 risk，必須在之後執行
    offer = await generate_offer(orders, risk)
    print(f"  [{time.time()-start:.2f}s] generate_offer 完成")
    
    total = time.time() - start
    print(f"\n⏱️  並行執行總時間: {total:.2f}s")
    print(f"   理論最小值: max(0.15, 0.40, 0.30) + 0.20 = 0.60s")
    print(f"   節省: ~{(1.05 - total) / 1.05 * 100:.0f}%")
    return {"profile": profile, "orders": orders, "risk": risk, "offer": offer}

print("=" * 50)
print("✅ 簡單並行（3 個獨立工具）")
print("=" * 50)
result = await simple_parallel("user_001")

## Part 3：DAG 依賴解析引擎（核心實作）

真實場景：工具之間有複雜的依賴關係，需要動態解析執行順序。

In [ ]:
# 模擬 LLM 回傳的 tool_calls（帶依賴宣告）
# 這是 LLM 輸出的格式，工程師設計執行引擎來解析

tool_calls_with_deps = [
    {
        "id": "t1",
        "tool": "get_user_profile",
        "args": {"user_id": "user_001"},
        "depends_on": []  # 無依賴，可立即執行
    },
    {
        "id": "t2",
        "tool": "get_order_history",
        "args": {"user_id": "user_001"},
        "depends_on": []  # 無依賴，可立即執行
    },
    {
        "id": "t3",
        "tool": "get_risk_score",
        "args": {"user_id": "user_001"},
        "depends_on": ["t1"],  # 依賴 t1（需要 profile 做精準評分）
    },
    {
        "id": "t4",
        "tool": "generate_offer",
        "args": {},
        "depends_on": ["t2", "t3"],  # 依賴 t2 和 t3
    }
]

print("LLM 宣告的 tool_calls 依賴關係：")
print("""
  t1 (get_user_profile, 150ms)
  t2 (get_order_history, 400ms)
     ↓ 並行
  t3 (get_risk_score, 300ms) ← 依賴 t1
  t4 (generate_offer, 200ms) ← 依賴 t2 + t3

執行層次：
  Tier 1: [t1, t2] → 無依賴，立即並行
  Tier 2: [t3]     → 等 t1 完成後執行
  Tier 3: [t4]     → 等 t2 + t3 都完成後執行
""")

In [ ]:
# ✅ DAG 依賴解析引擎

# 工具函數注冊表（Tool Registry）
TOOL_REGISTRY: dict[str, Callable] = {
    "get_user_profile": get_user_profile,
    "get_order_history": get_order_history,
    "get_risk_score": get_risk_score,
    "generate_offer": generate_offer,
}

def resolve_execution_tiers(tool_calls: list[dict]) -> list[list[str]]:
    """
    將 tool_calls 分組為執行層次（Execution Tiers）
    同一層的工具可以並行執行
    """
    # 建立 id → tool_call 的映射
    call_map = {tc["id"]: tc for tc in tool_calls}
    
    # 計算每個工具的層次
    tier_map: dict[str, int] = {}
    
    def get_tier(tool_id: str) -> int:
        if tool_id in tier_map:
            return tier_map[tool_id]
        deps = call_map[tool_id]["depends_on"]
        if not deps:
            tier_map[tool_id] = 0
        else:
            tier_map[tool_id] = max(get_tier(dep) for dep in deps) + 1
        return tier_map[tool_id]
    
    for tc in tool_calls:
        get_tier(tc["id"])
    
    # 按層次分組
    max_tier = max(tier_map.values())
    tiers = [[] for _ in range(max_tier + 1)]
    for tool_id, tier in tier_map.items():
        tiers[tier].append(tool_id)
    
    return tiers


async def execute_dag(tool_calls: list[dict]) -> dict[str, Any]:
    """
    根據 DAG 依賴關係，最大化並行執行工具
    返回 {tool_id: result} 的映射
    """
    call_map = {tc["id"]: tc for tc in tool_calls}
    results: dict[str, Any] = {}  # 存放每個工具的執行結果
    
    # 解析執行層次
    tiers = resolve_execution_tiers(tool_calls)
    
    print("📊 DAG 執行計畫：")
    for i, tier in enumerate(tiers):
        print(f"   Tier {i}: {tier} → 並行執行")
    print()
    
    start_total = time.time()
    
    for tier_idx, tier_tools in enumerate(tiers):
        tier_start = time.time()
        
        # 準備此層所有工具的協程
        async def run_tool(tool_id: str):
            tc = call_map[tool_id]
            tool_func = TOOL_REGISTRY[tc["tool"]]
            
            # 從已完成的結果中注入依賴的輸出
            kwargs = dict(tc["args"])  # 複製原始 args
            
            # 自動注入依賴工具的輸出
            for dep_id in tc["depends_on"]:
                dep_result = results.get(dep_id, {})
                dep_tool_name = call_map[dep_id]["tool"]
                # 依工具名稱注入對應的參數
                if dep_tool_name == "get_user_profile":
                    kwargs["profile"] = dep_result
                elif dep_tool_name == "get_order_history":
                    kwargs["order_history"] = dep_result
                elif dep_tool_name == "get_risk_score":
                    kwargs["risk"] = dep_result
            
            result = await tool_func(**kwargs)
            return tool_id, result
        
        # 並行執行此層所有工具
        tier_results = await asyncio.gather(
            *[run_tool(tid) for tid in tier_tools]
        )
        
        tier_elapsed = time.time() - tier_start
        
        # 儲存結果
        for tool_id, result in tier_results:
            results[tool_id] = result
        
        print(f"  ✅ Tier {tier_idx} 完成（{tier_tools}）— {tier_elapsed:.2f}s")
    
    total_elapsed = time.time() - start_total
    print(f"\n⏱️  DAG 並行執行總時間: {total_elapsed:.2f}s")
    print(f"   順序執行需要: 0.15+0.40+0.30+0.20 = 1.05s")
    print(f"   DAG 並行需要: max(0.15,0.40) + 0.30 + 0.20 = 0.90s")
    print(f"   節省: ~{(1.05 - total_elapsed) / 1.05 * 100:.0f}%")
    
    return results


print("=" * 50)
print("✅ DAG 依賴解析引擎執行")
print("=" * 50)

dag_results = await execute_dag(tool_calls_with_deps)

print("\n📋 執行結果：")
for tid, result in dag_results.items():
    print(f"  {tid}: {result}")

## Part 4：錯誤處理策略（Fail-fast vs Partial Success）

In [ ]:
# 模擬可能失敗的工具

async def flaky_risk_score(user_id: str, profile: dict = None) -> dict:
    """模擬不穩定的工具（50% 機率失敗）"""
    import random
    await asyncio.sleep(0.30)
    if random.random() < 0.5:
        raise Exception("風險評分服務暫時不可用（503）")
    return {"risk_score": 35, "risk_level": "LOW"}


# 策略 1：Fail-fast（一個失敗，全部停止）
async def strategy_fail_fast(user_id: str) -> dict:
    """所有工具必須都成功，任何一個失敗就整個流程停止"""
    try:
        profile, orders, risk = await asyncio.gather(
            get_user_profile(user_id),
            get_order_history(user_id),
            flaky_risk_score(user_id)
            # gather 預設：任一協程拋出例外，整個 gather 立即失敗
        )
        return {"profile": profile, "orders": orders, "risk": risk}
    except Exception as e:
        print(f"  ❌ Fail-fast 觸發: {e}")
        raise  # 向上傳播錯誤


# 策略 2：Partial Success（允許部分失敗，其他繼續）
async def strategy_partial_success(user_id: str) -> dict:
    """即使某個工具失敗，其他工具的結果仍然可用"""
    results = await asyncio.gather(
        get_user_profile(user_id),
        get_order_history(user_id),
        flaky_risk_score(user_id),
        return_exceptions=True  # ← 關鍵：把例外當作返回值而不是拋出
    )
    
    profile, orders, risk = results
    
    # 處理可能的失敗
    if isinstance(risk, Exception):
        print(f"  ⚠️  risk_score 失敗: {risk}，使用降級值")
        risk = {"risk_score": 50, "risk_level": "UNKNOWN"}  # 降級 fallback
    
    return {"profile": profile, "orders": orders, "risk": risk}


# 策略 3：Retry with Exponential Backoff
async def with_retry(coro_factory: Callable, max_retries: int = 2, base_delay: float = 0.5):
    """帶指數退避的重試包裝器"""
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return await coro_factory()
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"  🔄 重試 {attempt+1}/{max_retries}，等待 {delay:.1f}s...")
                await asyncio.sleep(delay)
    raise last_error


# 測試三種策略
print("=" * 50)
print("錯誤處理策略比較")
print("=" * 50)

# 策略 1：Fail-fast
print("\n策略 1: Fail-fast")
try:
    r = await strategy_fail_fast("user_001")
    print("  ✅ 成功", r.get("risk", {}).get("risk_level"))
except Exception as e:
    print(f"  ✅ 正確失敗（整個流程停止）")

# 策略 2：Partial Success
print("\n策略 2: Partial Success")
r2 = await strategy_partial_success("user_001")
print(f"  ✅ 完成，risk_level={r2['risk']['risk_level']}")

# 策略 3：Retry（對 Partial Success 的 risk 做重試）
print("\n策略 3: Retry with Backoff")
try:
    result = await with_retry(lambda: flaky_risk_score("user_001"), max_retries=2)
    print(f"  ✅ 重試後成功: {result}")
except Exception as e:
    print(f"  ❌ 重試失敗，使用 fallback: {e}")

## Part 5：延遲對比可視化 + 白板語言

In [ ]:
# 量化對比（多次執行取平均）

N_RUNS = 3

print("=" * 50)
print(f"延遲對比（{N_RUNS} 次平均）")
print("=" * 50)

# 順序執行
seq_times = []
for _ in range(N_RUNS):
    t = time.time()
    profile = await get_user_profile("u")
    orders = await get_order_history("u")
    risk = await get_risk_score("u", profile)
    offer = await generate_offer(orders, risk)
    seq_times.append(time.time() - t)

# 並行執行
par_times = []
for _ in range(N_RUNS):
    t = time.time()
    dag_res = await execute_dag(tool_calls_with_deps)
    par_times.append(time.time() - t)

seq_avg = sum(seq_times) / N_RUNS
par_avg = sum(par_times) / N_RUNS
savings = (seq_avg - par_avg) / seq_avg * 100

print(f"\n📊 結果：")
print(f"  順序執行平均: {seq_avg:.2f}s")
print(f"  DAG 並行平均: {par_avg:.2f}s")
print(f"  延遲節省: {savings:.0f}%")
print(f"\n面試量化語言：")
print(f"  『在 5 輪對話中，每輪有 3 個工具：")
print(f"    順序: 5 × {seq_avg:.2f}s = {5*seq_avg:.1f}s")
print(f"    並行: 5 × {par_avg:.2f}s = {5*par_avg:.1f}s")
print(f"    用戶體驗差距: {5*seq_avg - 5*par_avg:.1f}s』")

In [ ]:
# 面試 Trade-off 速查

print("面試 Trade-off 速查")
print("=" * 50)

print("""
Q: asyncio vs ThreadPoolExecutor，什麼時候選哪個？
   asyncio: I/O 密集型（HTTP API, DB 查詢）← 99% 的 Tool-calling 場景
   ThreadPoolExecutor: CPU 密集型 or 無法改成 async 的舊版 SDK

Q: 依賴關係讓 LLM 自己宣告 vs 程式碼 hardcode？
   LLM 宣告（動態）: 彈性高，新增工具不用改 Engine，但 LLM 可能宣告錯誤
   Hardcode（靜態）: 可靠，但每次新增工具都要改 DAG 定義
   推薦: 先 hardcode，等 LLM 的宣告準確率夠高再切換動態

Q: 錯誤策略選 Fail-fast 還是 Partial Success？
   Fail-fast: 所有工具結果缺一不可（財務報告、法律審閱）
   Partial Success: 部分工具是 optional（用於補充，不是核心）
   關鍵問題: 「如果工具 X 失敗，業務還能繼續嗎？"

Q: Google ADK 的 Tool Registry 能解決什麼？
   每個工具宣告 depends_on，ADK Orchestrator 自動管理並行和輸出注入
   省去手動寫 DAG Engine 的工程量（適合已在 GCP 的系統）
""")